# Analysing the Internals of Glove 100 angular dataset

The Glove 100 angular dataset is a collection of word embeddings that represent words in a high-dimensional space. Each word is represented as a vector of 100 dimensions, capturing semantic relationships between words based on their co-occurrence in large text corpora.

In [1]:
"""
This script is to use pandas to read and anaylze the sift-128-euclidean.hdf5 file. The file contains a dataset of SIFT descriptors, which are 128-dimensional feature vectors used in computer vision tasks. The script reads the data from the HDF5 file and converts it into a pandas DataFrame for further analysis.
"""

import h5py
import pandas as pd

FILE_PATH = './data/raw/glove-100-angular.hdf5'


with h5py.File(FILE_PATH, 'r') as f:
    # Read the data from the HDF5 file
    train_data = f['train'][:]
    test_data = f['test'][:]
    neighbors_data = f['neighbors'][:]
    distances_data = f['distances'][:]

# Convert the data to pandas DataFrames
train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)
neighbors_df = pd.DataFrame(neighbors_data)
distances_df = pd.DataFrame(distances_data)

# Perform your analysis here
print("Train data shape:", train_df.shape)
print("Test data shape:", test_df.shape)
print("Neighbors data shape:", neighbors_df.shape)
print("Distances data shape:", distances_df.shape)


Train data shape: (1183514, 100)
Test data shape: (10000, 100)
Neighbors data shape: (10000, 100)
Distances data shape: (10000, 100)


In [2]:
"""
This cell will deal with checking the data inside the hdf5 file and analyzing it using pandas. The data is stored in a hierarchical structure, and we can access the datasets using their keys. We will read the datasets into pandas DataFrames for easier analysis and manipulation. 
"""

data = {
    'train': train_df,
    'test': test_df,
    'neighbors': neighbors_df,
    'distances': distances_df
}

# show first 5 rows of each dataset
for key, df in data.items():
    print(f"\n{key} dataset:")
    print(df.head())


train dataset:
        0        1         2        3         4        5         6        7   \
0 -0.11333  0.48402  0.090771 -0.22439  0.034206 -0.55831  0.041849 -0.53573   
1 -0.99544 -2.36510 -0.243320 -1.03210  0.420520 -1.18170 -0.164510 -1.68300   
2 -0.10185  0.59817  0.202810  0.25243 -1.058300 -0.33025  1.515000 -0.31013   
3 -0.73278 -0.47153  1.331200  0.94514  0.391930 -0.15517  0.770010 -0.48571   
4 -0.21327  0.63622 -0.503040  0.17258 -0.537730 -0.51566 -0.499850 -1.02340   

        8        9   ...       90       91       92        93       94  \
0  0.18809 -0.58722  ... -0.17456  0.53571 -0.16537  0.135820 -0.56016   
1  0.49673 -0.27258  ...  0.40869 -0.08503  0.64130 -0.840440 -0.74325   
2 -0.52481  0.63273  ... -0.17440 -1.38970  0.37120  1.538800 -0.04959   
3 -0.25486 -0.51172  ...  0.46976 -0.70778  0.35542  1.015900  0.38979   
4 -0.35922 -1.39940  ... -0.26148 -0.66992 -0.58094  0.040179  0.33297   

         95        96        97        98        99  
0  0

## Understanding overall structure of the dataset

The dataset is organized in a hierarchical structure, with groups and datasets that store the word embeddings and associated metadata. The main components of the dataset include:
1. **Train data**: This contains the training set of word embeddings, which are used to learn the relationships between words.
2. **Test data**: This contains the test set of word embeddings, which are used to evaluate the performance of models trained on the training data.
3. **Neighbors data**: This contains the nearest neighbors for each word in the test set, which are used to evaluate the quality of the word embeddings.
4. **Distances data**: This contains the distances between each word in the test set and its nearest neighbors, which are used to evaluate the quality of the word embeddings.

In [ ]:
"""
This cell will be a random indexing implementation. For every query in the test dataset, we will randomly select a subset of the training dataset and compute the distances to find the nearest neighbors. This will help us understand the distribution of distances and the effectiveness of random indexing for approximate nearest neighbor search.
"""

# first we define a function to compute the Euclidean distance between two vectors
import numpy as np
import tqdm
import time

# for glove 100 angular dataset, we will use cosine distance instead of Euclidean distance
# formula for cosine distance is 1 - (A.B) / (||A|| * ||B||)
def cosine_distance(a, b):
    return 1 - np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# now we will randomly select a subset of 1000 samples from the training dataset for first 10 query in the test dataset, and return top 10 nearest neighbors for each query
# also calculate time of execution for each query and store it in a list
random_execution_times = []
random_nearest_neighbors_indices_list = list(list())
random_nearest_neighbors_distances_list = list(list())

for i in tqdm.tqdm(range(10)):
    query = test_df.iloc[i].values
    random_indices = np.random.choice(train_df.index, size=1000, replace=False)
    random_subset = train_df.iloc[random_indices].values
    # iloc is used to select rows by index, and values is used to get the underlying numpy array
    
    start_time = time.time()
    distances = np.array([cosine_distance(query, x) for x in random_subset])
    random_nearest_neighbors_indices = np.argsort(distances)[:10]
    # also return the distances of the nearest neighbors
    random_nearest_neighbors_distances = distances[random_nearest_neighbors_indices]
    random_execution_time = time.time() - start_time
    random_execution_times.append(random_execution_time)
    
    print(f"Query {i}: Nearest neighbors indices: {random_nearest_neighbors_indices}, Execution time: {random_execution_time:.4f} seconds")
    print(f"Query {i}: Nearest neighbors distances: {random_nearest_neighbors_distances}")
    random_nearest_neighbors_indices_list.append(random_nearest_neighbors_indices)
    random_nearest_neighbors_distances_list.append(random_nearest_neighbors_distances)
print(random_nearest_neighbors_indices_list)
print(random_nearest_neighbors_distances_list)


 60%|██████    | 6/10 [00:00<00:00, 26.07it/s]

Query 0: Nearest neighbors indices: [124 932 459 369 365 585 856 589 764 732], Execution time: 0.0040 seconds
Query 0: Nearest neighbors distances: [0.52932966 0.5297325  0.57760036 0.57803214 0.5905267  0.5917449
 0.59558374 0.6027187  0.60536367 0.6112678 ]
Query 1: Nearest neighbors indices: [416 871 229 712 384 265 145  33 479  64], Execution time: 0.0036 seconds
Query 1: Nearest neighbors distances: [0.41938198 0.5115577  0.5206841  0.54189867 0.54782104 0.5547854
 0.55986834 0.5758499  0.5815945  0.5834988 ]
Query 2: Nearest neighbors indices: [828 814 823 567 426 547 530 495 341  16], Execution time: 0.0037 seconds
Query 2: Nearest neighbors distances: [0.4251811  0.5665781  0.57106805 0.58154774 0.59638923 0.61348206
 0.6338906  0.64380133 0.644757   0.6532109 ]
Query 3: Nearest neighbors indices: [200 238  24 768 464 807  88 111 730  67], Execution time: 0.0039 seconds
Query 3: Nearest neighbors distances: [0.59247625 0.6011677  0.601867   0.6036888  0.6112268  0.61152434
 0.6

100%|██████████| 10/10 [00:00<00:00, 27.99it/s]

Query 7: Nearest neighbors indices: [863 422 170 122 998 360 152 576 603 217], Execution time: 0.0039 seconds
Query 7: Nearest neighbors distances: [0.5841369  0.58828944 0.59520435 0.5962031  0.6131913  0.6140554
 0.619784   0.6251745  0.6294094  0.63930786]
Query 8: Nearest neighbors indices: [219  29 747 936 120 932 637 342 887 947], Execution time: 0.0039 seconds
Query 8: Nearest neighbors distances: [0.6353389  0.6438589  0.6646058  0.66918266 0.66927266 0.66984963
 0.67023253 0.6927533  0.69587636 0.7117095 ]
Query 9: Nearest neighbors indices: [919 950 960 364 939 699 288 904  33 516], Execution time: 0.0039 seconds
Query 9: Nearest neighbors distances: [0.5070495  0.55207187 0.5831878  0.59166324 0.615345   0.62024033
 0.6232034  0.62641025 0.6353526  0.6394695 ]
[array([124, 932, 459, 369, 365, 585, 856, 589, 764, 732]), array([416, 871, 229, 712, 384, 265, 145,  33, 479,  64]), array([828, 814, 823, 567, 426, 547, 530, 495, 341,  16]), array([200, 238,  24, 768, 464, 807,  88

In [7]:
"""
This cell will be a brute force implementation. For every query in the test dataset, we will compute the distances to all samples in the training dataset and find the nearest neighbors. This will help us understand the distribution of distances and the effectiveness of brute force search for exact nearest neighbor search. This is similar to KNN search, but we will not use any libraries for this implementation. We will implement the distance calculation and nearest neighbor search from scratch.
"""

exact_execution_times = []
exact_nearest_neighbors_indices_list = list(list())
exact_nearest_neighbors_distances_list = list(list())

for i in tqdm.tqdm(range(10)):
    query = test_df.iloc[i].values
    start_time = time.time()
    distances = np.array([cosine_distance(query, x) for x in train_df.values])
    exact_nearest_neighbors_indices = np.argsort(distances)[:10]
    exact_nearest_neighbors_distances = distances[exact_nearest_neighbors_indices]
    exact_execution_time = time.time() - start_time
    exact_execution_times.append(exact_execution_time)
    
    print(f"Query {i}: Nearest neighbors indices: {exact_nearest_neighbors_indices}, Execution time: {exact_execution_time:.4f} seconds")
    print(f"Query {i}: Nearest neighbors distances: {exact_nearest_neighbors_distances}")
    exact_nearest_neighbors_indices_list.append(exact_nearest_neighbors_indices)
    exact_nearest_neighbors_distances_list.append(exact_nearest_neighbors_distances)

 10%|█         | 1/10 [00:04<00:39,  4.37s/it]

Query 0: Nearest neighbors indices: [  97478  262700  846101  671078  232287  727732  544474 1133489  723915
  660281], Execution time: 4.3690 seconds
Query 0: Nearest neighbors distances: [0.4303941  0.4323855  0.43309093 0.43341446 0.4375894  0.4397995
 0.4496429  0.4500963  0.45196807 0.4522282 ]


 20%|██        | 2/10 [00:08<00:34,  4.35s/it]

Query 1: Nearest neighbors indices: [875925 903728 144313 507188 547349 643514 663762 675600 891287 712921], Execution time: 4.3284 seconds
Query 1: Nearest neighbors distances: [0.3585065  0.36854464 0.386401   0.38654637 0.3876366  0.3932249
 0.39365864 0.39634246 0.3968156  0.39827985]


 30%|███       | 3/10 [00:13<00:30,  4.35s/it]

Query 2: Nearest neighbors indices: [1046944  809599  531832  323299 1077727  704557   62703  988527  377259
  625676], Execution time: 4.3521 seconds
Query 2: Nearest neighbors distances: [0.36257327 0.3861372  0.39548653 0.39753222 0.4052965  0.41076195
 0.41782224 0.42036742 0.4239397  0.4251811 ]


... (10 outputs left out)

 90%|█████████ | 9/10 [00:39<00:04,  4.37s/it]

Query 8: Nearest neighbors indices: [ 282309 1143087  839330  867615  402817   10347  787117  583277  379662
  356535], Execution time: 4.3904 seconds
Query 8: Nearest neighbors distances: [0.22978008 0.32861102 0.36337483 0.4038509  0.4228289  0.4288115
 0.43194228 0.43435186 0.4433967  0.4447456 ]


100%|██████████| 10/10 [00:43<00:00,  4.37s/it]

Query 9: Nearest neighbors indices: [505602 646337 317731 797071 496427 226657 493742 575987  44935 391014], Execution time: 4.4466 seconds
Query 9: Nearest neighbors distances: [0.4157973  0.44511503 0.44626427 0.45290726 0.45617008 0.45865387
 0.46201754 0.46445727 0.46969557 0.47084534]


In [8]:
K = 10
N_QUERIES = 10

# Ground truth (slice rows AND columns in one step)
true_idx = neighbors_df.iloc[:N_QUERIES, :K].values
true_dist = np.round(distances_df.iloc[:N_QUERIES, :K].values, 2)

def compare(name, idx_list, dist_list):
    idx = np.array(idx_list)[:N_QUERIES, :K]
    dist = np.round(np.array(dist_list)[:N_QUERIES, :K], 2)

    # Per-query results
    index_match = (idx == true_idx).astype(int)               # (N_QUERIES, K) 1 = same position
    dist_diff = np.abs(dist - true_dist).mean(axis=1)         # mean abs distance error per query

    for i in range(N_QUERIES):
        print(f"Query {i} | {name} vs true | index matches: {index_match[i].tolist()}")
        print(f"Query {i} | {name} vs true | mean abs distance diff: {dist_diff[i]:.4f}")

    return index_match, dist_diff

rand_idx_match, rand_dist_diff = compare(
    "random", random_nearest_neighbors_indices_list, random_nearest_neighbors_distances_list)
exact_idx_match, exact_dist_diff = compare(
    "exact", exact_nearest_neighbors_indices_list, exact_nearest_neighbors_distances_list)

# Store in a DataFrame for further analysis
results_df = pd.DataFrame({
    "query": range(N_QUERIES),
    "random_index_match_rate": rand_idx_match.mean(axis=1),
    "random_mean_abs_dist_diff": rand_dist_diff,
    "exact_index_match_rate": exact_idx_match.mean(axis=1),
    "exact_mean_abs_dist_diff": exact_dist_diff,
})
print(results_df)

Query 0 | random vs true | index matches: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Query 0 | random vs true | mean abs distance diff: 0.1420
Query 1 | random vs true | index matches: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Query 1 | random vs true | mean abs distance diff: 0.1510
Query 2 | random vs true | index matches: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Query 2 | random vs true | mean abs distance diff: 0.1860
Query 3 | random vs true | index matches: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Query 3 | random vs true | mean abs distance diff: 0.1570
Query 4 | random vs true | index matches: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Query 4 | random vs true | mean abs distance diff: 0.1620
Query 5 | random vs true | index matches: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Query 5 | random vs true | mean abs distance diff: 0.1500
Query 6 | random vs true | index matches: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Query 6 | random vs true | mean abs distance diff: 0.2880
Query 7 | random vs true | index matches: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Query 7 | 